# K-Means Clustering

The central question here: do LLM responses naturally group into quality archetypes, without us telling the algorithm what labels to look for?

We sweep K from 2 to 12, measure cluster quality three ways (inertia, silhouette, Davies-Bouldin), and let the data tell us how many natural groups exist. Whatever K wins gets its labels saved for the archetype analysis in notebook 03.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap

from inference_lens.clustering.cluster import run_kmeans
from inference_lens.utils.logging import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

import os
os.makedirs('../../reports/figures', exist_ok=True)
os.makedirs('../../data/processed', exist_ok=True)

## Load embeddings and features

In [ ]:
embeddings = np.load('../../data/processed/embeddings.npy')
df = pd.read_parquet('../../data/processed/features_with_labels.parquet')

print(f'Embeddings: {embeddings.shape}')
print(f'Features:   {df.shape}')
assert len(embeddings) == len(df), 'Row count mismatch between embeddings and features'

## K-Means sweep: K = 2 to 12

This takes a few minutes. Each K gets a full fit plus silhouette and Davies-Bouldin scores.

Note: silhouette is sampled at 5000 points for speed since the full dataset makes it prohibitively slow.

In [ ]:
results = run_kmeans(
    embeddings,
    k_range=range(2, 13),
    random_state=42,
    n_init=10,
)

# collect metrics into a tidy dataframe
metrics = pd.DataFrame([
    {
        'k': k,
        'inertia': v['inertia'],
        'silhouette': v['silhouette'],
        'davies_bouldin': v['davies_bouldin'],
    }
    for k, v in results.items()
])

metrics

## Elbow, silhouette, and Davies-Bouldin plots

Three lenses on the same question. The elbow is where inertia stops dropping fast. Silhouette peaks at the best K. Davies-Bouldin dips at the best K. Where they agree, that's your K.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(metrics['k'], metrics['inertia'], marker='o', color='steelblue')
axes[0].set_title('Elbow plot (inertia)')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_xticks(metrics['k'])

axes[1].plot(metrics['k'], metrics['silhouette'], marker='o', color='mediumseagreen')
axes[1].set_title('Silhouette score (higher is better)')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette')
axes[1].set_xticks(metrics['k'])

axes[2].plot(metrics['k'], metrics['davies_bouldin'], marker='o', color='tomato')
axes[2].set_title('Davies-Bouldin index (lower is better)')
axes[2].set_xlabel('K')
axes[2].set_ylabel('Davies-Bouldin')
axes[2].set_xticks(metrics['k'])

plt.suptitle('K-Means cluster quality metrics', fontsize=13)
plt.tight_layout()
plt.savefig('../../reports/figures/07_kmeans_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## Pick the optimal K

Set `optimal_k` based on what you see in the plots above. The silhouette peak and Davies-Bouldin dip are your primary signals. If they disagree, prefer silhouette.

In [ ]:
# set this after looking at the plots
optimal_k = int(metrics.loc[metrics['silhouette'].idxmax(), 'k'])
print(f'Auto-selected K = {optimal_k} based on silhouette peak')
print(f'Silhouette at K={optimal_k}: {metrics[metrics["k"]==optimal_k]["silhouette"].values[0]:.4f}')
print()
print('Full metrics table:')
print(metrics.to_string(index=False))

In [ ]:
# extract the labels for optimal K
kmeans_labels = results[optimal_k]['labels']
print(f'Cluster sizes at K={optimal_k}:')
unique, counts = np.unique(kmeans_labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f'  cluster {cluster_id}: {count:,} responses ({count/len(kmeans_labels)*100:.1f}%)')

## Visualize clusters in UMAP space

The same 5000-sample UMAP from EDA, now colored by cluster assignment. If K-Means found real structure, you should see cleanly separated regions.

In [ ]:
n_sample = 5000
idx = np.random.RandomState(42).choice(len(embeddings), n_sample, replace=False)
emb_sample = embeddings[idx]
labels_sample = kmeans_labels[idx]
pref_sample = df['label'].values[idx]

print('Fitting UMAP...')
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
emb_2d = reducer.fit_transform(emb_sample)
print('Done.')

In [ ]:
palette = sns.color_palette('tab10', n_colors=optimal_k)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# by cluster
for cluster_id in range(optimal_k):
    mask = labels_sample == cluster_id
    axes[0].scatter(
        emb_2d[mask, 0], emb_2d[mask, 1],
        s=4, alpha=0.5, color=palette[cluster_id],
        label=f'Cluster {cluster_id}'
    )
axes[0].set_title(f'K-Means clusters (K={optimal_k})')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')
axes[0].legend(markerscale=3, fontsize=8)

# by preference label
pref_colors = ['tomato' if l == 0 else 'steelblue' for l in pref_sample]
axes[1].scatter(emb_2d[:, 0], emb_2d[:, 1], c=pref_colors, s=4, alpha=0.5)
axes[1].set_title('Same points colored by preference label')
axes[1].set_xlabel('UMAP 1')

plt.suptitle('K-Means clusters vs preference labels in UMAP space', fontsize=13)
plt.tight_layout()
plt.savefig('../../reports/figures/08_kmeans_umap.png', dpi=150, bbox_inches='tight')
plt.show()

## Save cluster assignments

Attach K-Means labels to the features DataFrame and save. The archetype analysis notebook picks up from here.

In [ ]:
df['kmeans_cluster'] = kmeans_labels
df.to_parquet('../../data/processed/features_with_clusters.parquet', index=False)

np.save('../../data/processed/umap_2d.npy', emb_2d)
np.save('../../data/processed/umap_sample_idx.npy', idx)

print(f'Saved features_with_clusters.parquet with kmeans_cluster column (K={optimal_k})')
print(f'Saved UMAP coordinates for {n_sample} sample points')